# Tweezer Experiment

In [ ]:
import pytweezer.phasemask as pm
import pytweezer.communication as com
from pytweezer.experiment import motmaster_client
from pytweezer.analysis import analysis as pyt
from pytweezer.drivers.imagemX2 import ImagEMX2CameraClient, ImagEMX2Camera
from pytweezer.drivers import anapico
import matplotlib.pyplot as plt
import numpy as np
from time import sleep
import sys
from rich.progress import track
from scipy.signal import savgol_filter


texp = pyt.TweezerExperimentAnalysis(day='03',month='06Jun', year='2026')
exp = motmaster_client.MotMasterClient("Rb")
local_cam : ImagEMX2Camera = ImagEMX2Camera()

In [ ]:
local_cam.set_trigger_source("ext")
local_cam.set_external_exposure_mode()
local_cam.enable_em_gain(True)
local_cam.enable_direct_em_gain(True)
local_cam.set_sensitivity(1200)
local_cam.timeout = 60*20
X0, Y0, WIDTH, HEIGHT = 124, 170, 256, 256
local_cam.set_roi(X0, WIDTH, Y0, HEIGHT)

---

In [ ]:
dx, dy = 8.0, 4.0

In [ ]:
PM = pm.OptimisationBasedPhasemaskGeneratorGPU(
                 wavelength_um=0.852,
                 focal_length_mm=17.3,
                 slm_pitch_um=17,
                 slm_res=(1024,1024),
                 input_beam_waist_mm=16,
                 fresnel_f_mm=1072,
                 blaze_dx_dy_um=(40+dx, -8+dy),
                 zernike_coeff_dict={5:1.195, 6:0.725, 7:0.970, 8:0.478, 9:-1.091, 10:0.303, 11:0.021, 12:0.072, 13:0.049})

---
## Generate Initial Array Phasemask

In [ ]:
gauss2d = lambda x, y, A, x0, y0, sx, sy, offset: (A * np.exp(-(((x - x0) ** 2) / (2 * sx ** 2) + ((y - y0) ** 2) / (2 * sy ** 2)))) + offset
A, x0, y0, sx, sy, offset = [-2.14197604e+03, -1.11443850e+01 - dx, 2.09919600e+00 - dy, 2.95002910e+03, 2.64281503e+03, 2.14239505e+03]

In [ ]:
W1 = np.ones((16, 16))
spacing_um = 5.0

w_n, theta_n, x_n, y_n, array_shape = PM.generate_weighted_array(W1, spacing_um, init_phase_randomness=1.0, angle_deg=2.5)
w_n = (W1 * gauss2d(x_n.reshape(array_shape), y_n.reshape(array_shape), A, x0, y0, sx, sy, offset)).flatten()

terms1 = [w_n, theta_n, x_n, y_n, array_shape]
pm_array1, terms1, _ = PM.superposition_optimization(terms1, max_iter=2000, damping=0.5, verbose=True)

In [ ]:
SLM = com.SLMClient()
pm_composite = PM.superimpose([pm_array1, PM.fresnel, PM.blaze, PM.zernike])
pm_composite_uint8 = PM.transform_phase_8bit(pm_composite).get()
SLM.update_mask(pm_composite_uint8)

---
## Baseline Scan - Detect Site Positions


In [ ]:
# Experiment and camera setup
n_iterations = 100

exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)
local_cam.setup_acquisition("snap", n_iterations * 2)
local_cam.start_acquisition()
exp.start_motmaster_experiment()

imgs = local_cam.acquire_n_frames(n_iterations * 2)
imgs1 = imgs[::2]
imgs2 = imgs[1::2]
img_average = texp.tweezer_show_bg_subtracted(imgs1, imgs2, reg=[0, -1, 0, -1], cmap='gray', show=True, vmaxfactor=0.6, show_grid=True)

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

filtered_img_average = pyt.morphological_tophat_high_pass(img_average, feature_size=feature_size)
grid_positions, detection_threshold = pyt.detect_trap_sites(filtered_img_average, array_shape, detection_step=100)
pyt.visualize_results(filtered_img_average, grid_positions, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

imgs_filtered = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_filtered, grid_positions, array_shape, threshold_detection=True, window_size=window, binning=60)

In [ ]:
a = 0.00483372
b = 1828.38
photons = threshold*1000*80e-3
electrons = photons * 0.7
threshold_counts = electrons / a + b
print(threshold_counts)

In [ ]:
num_atoms_list = []
for pr in photon_rates:
    num_atoms = np.sum(pr > threshold)
    num_atoms_list.append(num_atoms)

# Fit a Gaussian distribution to num_atoms_list, extract the mean, and standard deviation.
mean_num_atoms = np.mean(num_atoms_list)
std_num_atoms = np.std(num_atoms_list)

# Plot a gaussian distribution to the data using the mean and standard deviation.
x = np.linspace(0, 256, 1000)
gaussian = (1 / (std_num_atoms * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean_num_atoms) / std_num_atoms) ** 2)

# In the legend include the mean and standard deviation of the fitted Gaussian distribution.
plt.hist(num_atoms_list, range=(0, 256), bins=65, density=True, alpha=0.7, color='gray', label='Data', edgecolor='black')
plt.plot(x, gaussian, color='red', linewidth=2, label='Gaussian Fit\n$\mu$={:.2f}\n$\sigma$={:.2f}'.format(mean_num_atoms, std_num_atoms))
plt.xlabel('Number of Atoms Loaded')
plt.ylabel('Probability Density')
plt.title('Histogram of Number of Atoms Loaded')
plt.legend()
plt.grid()

# Calculate the probability of obtaining fewer than 100 atoms loaded using the fitted Gaussian distribution.
from scipy.stats import norm

n_at = 10 ** 2
prob_fewer = norm.cdf(n_at, loc=mean_num_atoms, scale=std_num_atoms)

print(f"Filling fraction: {mean_num_atoms / 256:.2%}")
print(f"Probability of obtaining fewer than {n_at} atoms loaded: {prob_fewer * 100:.6f}%")
print(f"Probability of obtaining more than {n_at} atoms loaded: {(1 - prob_fewer) * 100:.6f}%")

## Baseline Survival Probability - 100 ms Rearrangement Delay

- In this case we DO NOT ramp down the power to 50%. We only do that when we are rearranging to a smaller array. Keep the set vva values for the tweezer power at 0.0.
- Imaging for 50 ms. Img cool VCO at 0.911 and VVA at 0.43.

In [ ]:
def extract_survival_probability(imgs1, imgs2):
    pr_1, eta_1, thresh_1, fidelity_1 = texp.get_array_loading_statistics(imgs1, grid_positions, array_shape, threshold_detection=True, window_size=window, binning=60, show_histogram=False, verbose=False)
    pr_2, eta_2, thresh_2, fidelity_2 = texp.get_array_loading_statistics(imgs2, grid_positions, array_shape, threshold_detection=True, window_size=window, binning=60, show_histogram=False, verbose=False)
    survival_fractions = []
    for i in range(pr_1.shape[0]):

        pr_mat_1 = pr_1[i]
        pr_mat_2 = pr_2[i]

        pr_mat_1_binary = (pr_mat_1 > thresh_1).astype(int)
        pr_mat_2_binary = (pr_mat_2 > thresh_2).astype(int)
        survival_matrix = pr_mat_1_binary * pr_mat_2_binary

        init_occupation = pr_mat_1_binary.sum()
        final_occupation = pr_mat_2_binary.sum()
        survival_count = survival_matrix.sum()
        survival_fraction = survival_count / init_occupation if init_occupation > 0 else 0
        survival_fractions.append(survival_fraction)

    survival_probability = np.mean(survival_fractions)
    return survival_probability

In [ ]:
# Experiment and camera setup
n_iterations = 100
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)
local_cam.setup_acquisition("snap", n_iterations * 2)
local_cam.start_acquisition()
exp.start_motmaster_experiment({"BackgroundDrop": 0})

imgs = local_cam.acquire_n_frames(n_iterations * 2)
imgs1, imgs2 = imgs[::2], imgs[1::2]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]
imgs_11_mean = np.mean(imgs_11, axis=0)
imgs_22_mean = np.mean(imgs_22, axis=0)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(imgs_11_mean, cmap='magma', vmax=imgs_11_mean.max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(imgs_22_mean, cmap='magma', vmax=imgs_22_mean.max()*0.6)
plt.title('Mean Image 2')
plt.show()

survival_prob = extract_survival_probability(imgs_11, imgs_22)
print(f"Survival Probability: {survival_prob * 100:.2f}%")

---
## Generate Final Array

In [ ]:
W2 = np.ones((10, 10))
spacing_um = 5.0

w_n, theta_n, x_n, y_n, array_shape = PM.generate_weighted_array(W2, spacing_um, init_phase_randomness=1.0, angle_deg=2.5)
w_n = (W2 * gauss2d(x_n.reshape(array_shape), y_n.reshape(array_shape), A, x0, y0, sx, sy, offset)).flatten()

terms2 = [w_n, theta_n, x_n, y_n, array_shape]
pm_array2, terms2, _ = PM.superposition_optimization(terms2, max_iter=2000, damping=0.5, verbose=True)

---
## Rearrangement Exp

In [ ]:
import asyncio
from pytweezer.communication import RearrangementNode

local_cam.relinquish_camera()
RN = RearrangementNode()

### Exp 1

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.2, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 200
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

In [ ]:
n = 0
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 30
threshold_bin = 0.25

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=100)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=True, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 66
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 20
threshold_bin = 0.3

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 2

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.2, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 100
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=False, threshold=0.5, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 24
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 25
threshold_bin = 0.32

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 3

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.00, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 100
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=False, threshold=0.5, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 24
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 25
threshold_bin = 0.32

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 4

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.00, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 200
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=False, threshold=0.5, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 138
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 25
threshold_bin = 0.30

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.show()

In [ ]:
n = 131
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(imgs_11[n], cmap='magma', vmax=imgs_11[n].max()*0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(imgs_22[n], cmap='magma', vmax=imgs_22[n].max()*0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 5

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.2, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 200
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=True, threshold=0.5, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 10
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 30
threshold_bin = 0.25

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.show()

In [ ]:
n = 10
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(imgs_11[n], cmap='magma', vmax=imgs_11[n].max()*0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(imgs_22[n], cmap='magma', vmax=imgs_22[n].max()*0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 6

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.0, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 200
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=False, threshold=1.0, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 5
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 30
threshold_bin = 0.25

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.show()

In [ ]:
n = 5
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(imgs_11[n], cmap='magma', vmax=imgs_11[n].max()*0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(imgs_22[n], cmap='magma', vmax=imgs_22[n].max()*0.6)
plt.title(f'Frame {n}')
plt.show()

### Exp 7

In [ ]:
await RN.initialise(terms1, terms2, grid_positions, threshold_counts, d0=1.0, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

# Experiment and camera setup
n_iterations = 1
exp.set_motmaster_experiment("RbTweezerBasic2026_2")
exp.set_iterations(n_iterations)
exp.set_save_toggle(False)
exp.set_run_until_stopped(False)

In [ ]:
N_iterations = 200
imgs = []
err_indices = []
occ_extraction_times = []
rearrangement_sequence_calc_times = []
slm_upload_times = []
total_rearrangement_times = []

for it in range(N_iterations):
    print(f"Running experiment iteration {it+1} / {N_iterations}")
    rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

    sleep(0.1)
    exp.start_motmaster_experiment({"BackgroundDrop": 0})
    img0, img1, timings = await rearrangement_task
    print("Images acquired from rearrangement node.")

    if img1.sum() == 0:
        print(f"Empty image received at iteration {it+1}. Discarding...")
        err_indices.append(it)
    else:
        imgs.append([img0, img1])

    occ_extraction_times.append(timings['occupancy_extraction_s'])
    rearrangement_sequence_calc_times.append(timings['rearrangement_sequence_generation_s'])
    slm_upload_times.append(timings['slm_upload_s'])
    total_rearrangement_times.append(timings['total_rearrangement_s'])

print(f"\nAverage occupancy extraction time = {np.mean(occ_extraction_times)*1000:.2f} ms")
print(f"Average rearrangement sequence generation time = {np.mean(rearrangement_sequence_calc_times)*1000:.2f} ms")
print(f"Average SLM upload time = {np.mean(slm_upload_times)*1000:.2f} ms")
print(f"Average total rearrangement time = {np.mean(total_rearrangement_times)*1000:.2f} ms")

In [ ]:
tot_imgs = np.array(imgs)
imgs1 = tot_imgs[:, 0]
imgs2 = tot_imgs[:, 1]
imgs_11 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
imgs_22 = [pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(np.mean(imgs_11, axis=0), cmap='magma', vmax=np.mean(imgs_11, axis=0).max()*0.6)
plt.title('Mean Image 1')
plt.subplot(1, 2, 2)
plt.imshow(np.mean(imgs_22, axis=0), cmap='magma', vmax=np.mean(imgs_22, axis=0).max()*0.6)
plt.title('Mean Image 2')
plt.show()

#### Initial Array Statistics

In [ ]:
array_shape = [16, 16]
window = 3
feature_size = 10

grid_positions11, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_11, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_11, axis=0), grid_positions11, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_11, grid_positions11, array_shape, threshold_detection=True, window_size=window, binning=60)

#### Final Array Statistics

In [ ]:
array_shape = [10, 10]
window = 3
feature_size = 10

grid_positions22, detection_threshold = pyt.detect_trap_sites(np.mean(imgs_22, axis=0), array_shape, detection_step=500)
pyt.visualize_results(np.mean(imgs_22, axis=0), grid_positions22, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.7)

photon_rates, loading_probabilities, threshold, fidelity = texp.get_array_loading_statistics(imgs_22, grid_positions22, array_shape, threshold_detection=False, threshold=1.2, window_size=window, binning=60)

In [ ]:
best_it, max_atoms = 0, 0
for it in range(photon_rates.shape[0]):
    pr_mat = photon_rates[it]
    pr_mat_binary = (pr_mat > threshold).astype(int)
    atom_count = pr_mat_binary.sum()
    print(f"Iteration {it+1}: Detected {atom_count} atoms.")
    if atom_count > max_atoms:
        max_atoms = atom_count
        best_it = it

print(f"Best iteration: {best_it+1} with {max_atoms} detected atoms.")

In [ ]:
n = 20
img1 = (imgs_11[n] - imgs_11[n].min()) / (imgs_11[n].max() - imgs_11[n].min())  
img2 = (imgs_22[n] - imgs_22[n].min()) / (imgs_22[n].max() - imgs_22[n].min())

sigmoid = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
bin_sharpness = 30
threshold_bin = 0.25

img1 = sigmoid(img1, bin_sharpness, threshold_bin)
img2 = sigmoid(img2, bin_sharpness, threshold_bin)

# Apply a filter to remove random speckle noise
from scipy.ndimage import median_filter
img1 = median_filter(img1, size=2)
img2 = median_filter(img2, size=2)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='magma', vmax=0.5)
plt.title(f'Frame {n}')
plt.show()

In [ ]:
n = 5
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(imgs_11[n], cmap='magma', vmax=imgs_11[n].max()*0.6)
plt.title(f'Frame {n}')
plt.subplot(1, 2, 2)
plt.imshow(imgs_22[n], cmap='magma', vmax=imgs_22[n].max()*0.6)
plt.title(f'Frame {n}')
plt.show()

---
## Rearrangement Animation

In [ ]:
import lap
import cupy as cp


def get_jv_pairing_lap(init, final):
        """
        Solves using the 'lap' library (C++ Jonker-Volgenant implementation).
        Computes the cost matrix and padding efficiently on the GPU via CuPy.
        """
        N = len(init)
        M = len(final)
        
        # 1. Calculate Cost Matrix directly on GPU using CuPy broadcasting
        # init shape: (N, 2), final shape: (M, 2) -> cost_matrix shape: (N, M)
        cost_matrix = cp.sum((init[:, None, :] - final[None, :, :])**2, axis=-1)
        
        # 2. Handle Rectangularity (N != M) via padding
        if N != M:
            dim = max(N, M)
            # Create a large square matrix on the GPU filled with a high cost
            large_cost = float(cost_matrix.max() * 1000.0) if cost_matrix.size > 0 else 1.0
            padded_cost = cp.full((dim, dim), large_cost, dtype=cp.float32)
            
            # Fill in the real data
            padded_cost[:N, :M] = cost_matrix
            
            # TRANSFER TO CPU FOR C++ LAPJV (lap strictly requires numpy)
            padded_cost_cpu = padded_cost.get()
            opt_cost, x, y = lap.lapjv(padded_cost_cpu, extend_cost=True)
            
            # 3. Extract valid indices and map back to CuPy
            if N < M:
                init_idx = cp.arange(N)
                final_idx = cp.asarray(x[:N])
                
                valid_mask = final_idx < M
                init_idx = init_idx[valid_mask]
                final_idx = final_idx[valid_mask]
            else:
                final_idx = cp.arange(M)
                init_idx = cp.asarray(y[:M])
                
                valid_mask = init_idx < N
                final_idx = final_idx[valid_mask]
                init_idx = init_idx[valid_mask]
                
        else:
            # Square case is simple
            cost_matrix_cpu = cost_matrix.get()
            opt_cost, x, y = lap.lapjv(cost_matrix_cpu, extend_cost=True)
            init_idx = cp.arange(N)
            final_idx = cp.asarray(x)
            
        return init_idx, final_idx

In [ ]:
d0 = 0.75

w1, phi1, x1, y1, arr_shape1 = terms1
w2, phi2, x2, y2, arr_shape2 = terms2

pos1 = cp.stack((x1, y1), axis=-1)
pos2 = cp.stack((x2, y2), axis=-1)

# INITIALIZE VRAM STATE MACHINE
curr_w = cp.asarray(w1, dtype=cp.float32)
curr_phi = cp.asarray(phi1, dtype=cp.float32)
curr_x = cp.asarray(x1, dtype=cp.float32)
curr_y = cp.asarray(y1, dtype=cp.float32)

# Initialize step vectors with zeros
dw = cp.zeros_like(curr_w)
dphi = cp.zeros_like(curr_phi)
total_dx = cp.zeros_like(curr_x)
total_dy = cp.zeros_like(curr_y)

# Ensure array operands are on the GPU to avoid implicit CPU conversion
w1_gpu, w2_gpu = cp.asarray(w1), cp.asarray(w2)
phi1_gpu, phi2_gpu = cp.asarray(phi1), cp.asarray(phi2)

occ_mask = cp.random.choice([True, False], size=(16, 16)).flatten()

occ_indices = cp.where(occ_mask)[0]
init = pos1[occ_indices]
final = pos2

init_idx, final_idx = get_jv_pairing_lap(init, final)

# Map the Hungarian output back to the original array indices
moving_idx = occ_indices[init_idx]

# Compute mask for traps to be switched off
off_mask = cp.ones(len(pos1), dtype=bool)
off_mask[moving_idx] = False

# 2. CALCULATE INTERPOLATION STEPS
pos_init = pos1[moving_idx]
pos_final = pos2[final_idx]
vec = pos_final - pos_init

# Faster L2 norm calculation using cupy linear algebra
max_dist = cp.linalg.norm(vec, axis=1).max()
#n_steps = int(cp.ceil(1.875 * max_dist / d0))          # Minimum jerk profile
n_steps = int(cp.ceil(max_dist / d0))                   # Linear profile

# Pre-calculate the minimum jerk step multipliers on the GPU
tau = cp.linspace(0, 1, n_steps + 1, dtype=cp.float32)  
#s_profile = 10 * tau**3 - 15 * tau**4 + 6 * tau**5     # Minimum jerk profile
s_profile = tau                                         # Linear profile

# ds_profile contains the fractional progression for each step n
ds_profile = cp.diff(s_profile)

# CRITICAL: Pull ds_profile back to the CPU! 
# Accessing GPU array scalars inside a loop causes implicit device synchronizations.
ds_profile_cpu = ds_profile.get()

# Load steps for MOVING traps
dw[moving_idx] = (w2_gpu[final_idx] - w1_gpu[moving_idx]) / n_steps
total_dx[moving_idx] = vec[:, 0].astype(cp.float32)
total_dy[moving_idx] = vec[:, 1].astype(cp.float32)

# Ensure Phase Interpolation takes the shortest angular path to prevent wrapping tears
phase_diff = (phi2_gpu[final_idx] - phi1_gpu[moving_idx] + cp.pi) % (2 * cp.pi) - cp.pi
dphi[moving_idx] = phase_diff / n_steps

# Load steps for OFF traps (Ramp down weights to 0)
dw[off_mask] = 0.0
curr_w[off_mask] = 0.0


In [ ]:
N_iterations = 100
final_images = []

for n in range(n_steps):
    print(f"Acquiring frame {n+1} / {n_steps}")

    ds = float(ds_profile_cpu[n])

    curr_x += total_dx * ds
    curr_y += total_dy * ds
    curr_w += dw
    curr_phi += dphi
    curr_terms = [curr_w, curr_phi, curr_x, curr_y, [16, 16]]

    await RN.initialise(terms1, curr_terms, grid_positions, threshold_counts, d0=1.0, fps=1/(1e-6), roi=[X0, Y0, WIDTH, HEIGHT])

    imgs = []
    for it in range(N_iterations):
        print(f"Running experiment iteration {it+1} / {N_iterations}")
        rearrangement_task = asyncio.create_task(RN.arm_rearrangement())

        sleep(0.2)
        exp.start_motmaster_experiment({"BackgroundDrop": 0})
        img0, img1, timings = await rearrangement_task
        print("Images acquired from rearrangement node.")

        if img1.sum() == 0:
            print(f"Empty image received at iteration {it+1}. Discarding...")
            err_indices.append(it)
        else:
            imgs.append([img0, img1])

    tot_imgs = np.array(imgs)
    imgs1, imgs2 = tot_imgs[:, 0], tot_imgs[:, 1]
    final_images.append(imgs2)


In [ ]:
frames = []

initial_frame = np.array([pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]).mean(axis=0)
for _ in range(5):
    frames.append(initial_frame)

for imgs2 in final_images:
    imgs_22 = np.array([pyt.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs2]).mean(axis=0)
    frames.append(imgs_22)

In [ ]:
# Frames is a list of images. Create an interactive animation to visualize the rearrangement progression using HTML and JavaScript.
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

def create_animation(frames, interval=200):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(frames[0], cmap='magma', vmax=np.array(frames).max()*0.6)
    plt.close(fig)

    def update(frame):
        im.set_data(frame)
        return [im]

    anim = FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)
    return HTML(anim.to_jshtml())

create_animation(frames)
